# Cafe Sales Analysis

Sales analysis of a cafe dataset to understand revenue, products, monthly sales, payments, and locations.

## 1. Import Libraries

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import plotly.express as px

## 2. Load Dataset

In [2]:
# Load the raw dataset
df = pd.read_csv("dirty_cafe_sales.csv")

# View the first five rows
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


## 3. Initial Data Check

In [3]:
# Check the dataset size
df.shape

(10000, 8)

In [4]:
# Check column names and data types
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
# Check missing values
df.isna().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

## 4. Data Cleaning

In [6]:
# Replace invalid text values with missing values
df = df.replace(["UNKNOWN", "ERROR"], np.nan)

In [7]:
# Fill missing product names with "Unknown"
df["Item"] = df["Item"].fillna("Unknown")

In [8]:
# Convert numeric columns to numbers
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price Per Unit"] = pd.to_numeric(df["Price Per Unit"], errors="coerce")
df["Total Spent"] = pd.to_numeric(df["Total Spent"], errors="coerce")

In [9]:
# Calculate missing Quantity where Price and Total Spent are available
calculated_quantity = (
    df["Total Spent"] / df["Price Per Unit"]
)

df["Quantity"] = df["Quantity"].fillna(calculated_quantity)

In [10]:
# Calculate missing Price Per Unit where Quantity and Total Spent are available
calculated_price = (
    df["Total Spent"] / df["Quantity"]
)

df["Price Per Unit"] = df["Price Per Unit"].fillna(calculated_price)

In [11]:
# Calculate missing Total Spent where Quantity and Price are available
calculated_total = (
    df["Quantity"] * df["Price Per Unit"]
)

df["Total Spent"] = df["Total Spent"].fillna(calculated_total)

In [12]:
# Fill missing payment and location values
df["Payment Method"] = df["Payment Method"].fillna("Unknown")
df["Location"] = df["Location"].fillna("Unknown")

In [13]:
# Convert Transaction Date to datetime
df["Transaction Date"] = pd.to_datetime(
    df["Transaction Date"],
    errors="coerce"
)

In [14]:
# Create a month column for monthly analysis
df["Month"] = df["Transaction Date"].dt.month

## 5. Data Validation

In [15]:
# Check the final missing values
df.isna().sum()

Transaction ID        0
Item                  0
Quantity             38
Price Per Unit       38
Total Spent          40
Payment Method        0
Location              0
Transaction Date    460
Month               460
dtype: int64

In [16]:
# Check for zero or negative values
invalid_values = {
    "Quantity": (df["Quantity"] <= 0).sum(),
    "Price Per Unit": (df["Price Per Unit"] <= 0).sum(),
    "Total Spent": (df["Total Spent"] <= 0).sum()
}

invalid_values

{'Quantity': np.int64(0),
 'Price Per Unit': np.int64(0),
 'Total Spent': np.int64(0)}

In [17]:
# Check that Total Spent matches Quantity × Price Per Unit
check = df[["Quantity", "Price Per Unit", "Total Spent"]].dropna()

(check["Quantity"] * check["Price Per Unit"] != check["Total Spent"]).sum()

np.int64(0)

## 6. Key Business Metrics

In [18]:
# Calculate key business metrics
company_kpis = {
    "Total Revenue": df["Total Spent"].sum(),
    "Total Transactions": df["Transaction ID"].nunique(),
    "Total Units Sold": df["Quantity"].sum(),
    "Average Transaction Value": df["Total Spent"].mean()
}

company_kpis

{'Total Revenue': np.float64(88952.0),
 'Total Transactions': 10000,
 'Total Units Sold': np.float64(30141.0),
 'Average Transaction Value': np.float64(8.930923694779116)}

## 7. Product Performance

In [19]:
# Calculate revenue by product
revenue_by_item = df.groupby("Item")["Total Spent"].sum()
revenue_by_item = revenue_by_item.sort_values(ascending=False)

revenue_by_item

Item
Salad       17320.0
Sandwich    13664.0
Smoothie    13320.0
Juice       10509.0
Cake        10395.0
Unknown      8507.5
Coffee       7062.0
Tea          4951.5
Cookie       3223.0
Name: Total Spent, dtype: float64

In [20]:
# Plot revenue by product
fig = px.bar(
    revenue_by_item,
    title="Revenue by Product",
    labels={"value": "Total Revenue"}
)
fig.show()

In [21]:
# Calculate units sold by product
units_sold_by_item = df.groupby("Item")["Quantity"].sum()
units_sold_by_item = units_sold_by_item.sort_values(ascending=False)

units_sold_by_item

Item
Coffee      3534.0
Juice       3505.0
Salad       3468.0
Cake        3462.0
Sandwich    3424.0
Smoothie    3330.0
Tea         3292.0
Cookie      3228.0
Unknown     2898.0
Name: Quantity, dtype: float64

In [22]:
# Plot units sold by product
fig = px.bar(
    units_sold_by_item,
    title="Units Sold by Product",
    labels={"value": "Units Sold"}
)
fig.show()

## 8. Monthly Sales

In [23]:
# Calculate monthly revenue
monthly_revenue = df.groupby("Month")["Total Spent"].sum()

monthly_revenue

Month
1.0     7242.0
2.0     6633.5
3.0     7214.5
4.0     7168.0
5.0     6941.5
6.0     7350.0
7.0     6877.5
8.0     7077.5
9.0     6846.0
10.0    7302.0
11.0    6957.0
12.0    7177.0
Name: Total Spent, dtype: float64

In [24]:
# Plot monthly revenue
fig = px.line(
    monthly_revenue,
    title="Monthly Revenue Trend",
    labels={"Month": "Month", "value": "Total Revenue"},
    markers=True
)
fig.show()

## 9. Payment Performance

In [25]:
# Calculate revenue by payment method
revenue_by_payment = df.groupby("Payment Method")["Total Spent"].sum()
revenue_by_payment = revenue_by_payment.sort_values(ascending=False)

revenue_by_payment

Payment Method
Unknown           27775.0
Credit Card       20427.0
Digital Wallet    20383.5
Cash              20366.5
Name: Total Spent, dtype: float64

In [26]:
# Exclude Unknown to compare known payment methods
known_payment_revenue = revenue_by_payment.drop("Unknown")

# Find the highest-revenue known payment method
top_known_payment = known_payment_revenue.idxmax()

top_known_payment

'Credit Card'

In [27]:
# Plot revenue by payment method
fig = px.bar(
    known_payment_revenue,
    title="Revenue by Payment Method",
    labels={"value": "Total Revenue"}
)
fig.show()

## 10. Location Performance

In [28]:
# Calculate revenue by location
revenue_by_location = df.groupby("Location")["Total Spent"].sum()
revenue_by_location = revenue_by_location.sort_values(ascending=False)

revenue_by_location

Location
Unknown     35337.5
In-store    27127.0
Takeaway    26487.5
Name: Total Spent, dtype: float64

In [29]:
# Exclude Unknown to compare known locations
known_location_revenue = revenue_by_location.drop("Unknown")

# Find the highest-revenue known location
top_known_location = known_location_revenue.idxmax()

top_known_location

'In-store'

In [30]:
# Plot revenue by location
fig = px.bar(
    known_location_revenue,
    title="Revenue by Location",
    labels={"value": "Total Revenue"}
)
fig.show()

## 11. Transaction Behavior

In [31]:
# Calculate average spending for each quantity level
avg_spent_by_quantity = df.groupby("Quantity")["Total Spent"].mean()

avg_spent_by_quantity

Quantity
1.0     2.957009
2.0     5.822494
3.0     8.883308
4.0    11.748837
5.0    14.863225
Name: Total Spent, dtype: float64

In [32]:
# Plot Quantity vs Total Spent
fig = px.scatter(
    df,
    x="Quantity",
    y="Total Spent",
    title="Quantity vs Total Spent"
)
fig.show()

## 12. Key Findings

- 10,000 transactions generated 88,952 in available revenue.
- Coffee sold the most units at 3,534.
- Salad generated the highest product revenue at 17,320.
- June had the highest monthly revenue at 7,350.
- Credit Card generated the highest revenue among known payment methods.
- In-store generated the highest revenue among known locations.
- Transaction Date has the highest missing data at 4.6%.

## 13. Recommendations

- Focus on high-revenue products such as Salad.
- Track monthly sales to understand changes in demand.
- Improve payment data collection to support better analysis.